# Evaluation Results Analysis

Loads `scores_raw.json` produced by `evaluation/llm_judge.py`.
Sources and colors are **auto-discovered** from the data — no code changes needed when adding new models.

Sections:
1. Load & configure
2. Parse into DataFrame
3. Summary table (mean ± std)
4. Radar chart
5. Bar chart per metric
6. Box plots across runs
7. Heatmap per (case × source)
8. Overall ranking
9. Pipetly model comparison
10. Score stability across runs
11. Justifications viewer

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1 · Load & configure

In [ ]:
# ── Point this to your scores_raw.json (or the results/ directory) ─────────────
RESULTS_PATH = Path("../evaluation/results")

if RESULTS_PATH.is_dir():
    folders = sorted([p for p in RESULTS_PATH.iterdir() if p.is_dir()], reverse=True)
    RESULTS_FILE = folders[0] / "scores_raw.json" if folders else next(RESULTS_PATH.glob("*.json"))
else:
    RESULTS_FILE = RESULTS_PATH

print(f"Loading: {RESULTS_FILE}")
data = json.loads(RESULTS_FILE.read_text(encoding="utf-8"))
print(f"Judge model : {data['judge_model']}")
print(f"Runs        : {data['n_runs']}")
print(f"Entries     : {data['n_entries']}")

# ── Metric constants ───────────────────────────────────────────────────────────
METRICS = [
    "relevance", "completeness", "parameter_consistency",
    "executability", "structural_coherence", "conciseness",
]
METRIC_LABELS = {
    "relevance": "Relevance",
    "completeness": "Completeness",
    "parameter_consistency": "Param. Consistency",
    "executability": "Executability",
    "structural_coherence": "Struct. Coherence",
    "conciseness": "Conciseness",
}

# ── Optional: override display names for specific sources ──────────────────────
# Leave empty to use the raw source key as label.
CUSTOM_LABELS: dict[str, str] = {
    # "pipetly_model1": "Pipetly (GPT-4o mini)",
    # "pipetly_model2": "Pipetly (GPT-4o)",
    # "gemini_pro":     "Gemini Pro (direct)",
    # "bioprobench":    "BioProBench",
}

## 2 · Parse into a flat DataFrame

In [ ]:
rows = []
for run in data["runs"]:
    for sr in run["results"]:
        if sr.get("error") or sr.get("result") is None:
            continue
        r = sr["result"]
        row = {"run": sr["run"], "entry_id": sr["entry_id"], "source": sr["source"]}
        for m in METRICS:
            row[m] = r[m]["score"]
            row[f"{m}_just"] = r[m]["justification"]
        row["mean"] = sum(r[m]["score"] for m in METRICS) / len(METRICS)
        rows.append(row)

df = pd.DataFrame(rows)

# ── Auto-discover sources in insertion order ───────────────────────────────────
seen = set()
sources_present = []
for run in data["runs"]:
    for sr in run["results"]:
        s = sr["source"]
        if s not in seen:
            seen.add(s)
            sources_present.append(s)

# ── Auto-generate labels & palette ────────────────────────────────────────────
def _label(src: str) -> str:
    return CUSTOM_LABELS.get(src, src.replace("_", " ").title())

_cmap = cm.get_cmap("tab10", max(len(sources_present), 1))
PALETTE = {src: _cmap(i) for i, src in enumerate(sources_present)}

df["source_label"] = df["source"].map(_label)
print(f"Rows   : {len(df)}")
print(f"Sources: {sources_present}")
df.head()

## 3 · Summary statistics

In [ ]:
score_cols = METRICS + ["mean"]
summary = df.groupby("source")[score_cols].agg(["mean", "std"])

pretty = {}
for src in sources_present:
    if src not in summary.index:
        continue
    pretty[_label(src)] = {
        METRIC_LABELS.get(m, m): (
            f"{summary.loc[src, (m, 'mean')]:.2f} ± {summary.loc[src, (m, 'std')]:.2f}"
            if not math.isnan(summary.loc[src, (m, 'std')])
            else f"{summary.loc[src, (m, 'mean')]:.2f}"
        )
        for m in score_cols
    }

summary_df = pd.DataFrame(pretty).T
summary_df.index.name = "Source"
summary_df

## 4 · Radar chart

In [ ]:
N = len(METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for src in sources_present:
    sub = df[df["source"] == src]
    if sub.empty:
        continue
    vals = sub[METRICS].mean().tolist() + [sub[METRICS].mean().tolist()[0]]
    ax.plot(angles, vals, color=PALETTE[src], linewidth=2, label=_label(src))
    ax.fill(angles, vals, color=PALETTE[src], alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], fontsize=10)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_ylim(0, 5)
ax.set_yticklabels(["1", "2", "3", "4", "5"], fontsize=8, color="grey")
ax.set_title("Protocol Quality — Radar Chart (mean across runs)",
             pad=20, fontsize=12, fontweight="bold")
ax.legend(loc="upper right", bbox_to_anchor=(1.5, 1.15), fontsize=9)
plt.tight_layout()
plt.show()

## 5 · Bar chart per metric (mean ± std)

In [ ]:
n_sources = len(sources_present)
x = np.arange(len(METRICS))
width = min(0.8 / n_sources, 0.25)

fig, ax = plt.subplots(figsize=(14, 5))

for i, src in enumerate(sources_present):
    sub = df[df["source"] == src]
    if sub.empty:
        continue
    means = [sub[m].mean() for m in METRICS]
    stds  = [sub[m].std()  for m in METRICS]
    offset = (i - (n_sources - 1) / 2) * width
    ax.bar(x + offset, means, width, yerr=stds, capsize=4,
           color=PALETTE[src], label=_label(src), alpha=0.85,
           error_kw=dict(elinewidth=1.2, ecolor="#333"))

ax.set_xticks(x)
ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], fontsize=10)
ax.set_ylabel("Score (1–5)", fontsize=11)
ax.set_ylim(0, 5.8)
ax.axhline(y=5, color="#ccc", linestyle="--", linewidth=0.8)
ax.set_title("Score per Metric — Mean ± Std across all runs", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6 · Box plots — distribution across runs

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(18, 4), sharey=True)

for ax, metric in zip(axes, METRICS):
    bp = ax.boxplot(
        [df[df["source"] == src][metric].values for src in sources_present],
        patch_artist=True,
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
    )
    for patch, src in zip(bp["boxes"], sources_present):
        patch.set_facecolor(PALETTE[src])
        patch.set_alpha(0.75)

    ax.set_title(METRIC_LABELS[metric], fontsize=9, fontweight="bold")
    ax.set_xticks(range(1, n_sources + 1))
    ax.set_xticklabels([_label(s) for s in sources_present],
                       rotation=30, ha="right", fontsize=7)
    ax.set_ylim(0.5, 5.5)
    ax.set_yticks([1, 2, 3, 4, 5])

axes[0].set_ylabel("Score (1–5)", fontsize=10)
fig.suptitle("Score Distribution per Metric across Runs",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 7 · Heatmap — mean score per (case × source)

In [ ]:
pivot = df.pivot_table(
    index=["entry_id", "source"], values=METRICS + ["mean"], aggfunc="mean"
).round(2)

col_labels = [METRIC_LABELS[m] for m in METRICS] + ["Overall mean"]
mat = pivot[METRICS + ["mean"]].values

fig, ax = plt.subplots(figsize=(14, max(3, len(pivot) * 0.5)))
im = ax.imshow(mat, cmap="YlGn", vmin=1, vmax=5, aspect="auto")

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(
    [f"{eid} · {_label(src)}" for eid, src in pivot.index], fontsize=8)

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center",
                fontsize=8, color="black")

plt.colorbar(im, ax=ax, label="Score (1–5)", fraction=0.02)
ax.set_title("Mean Scores per Case × Source", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 8 · Overall mean — ranking

In [ ]:
overall = (
    df.groupby("source")["mean"]
    .agg(["mean", "std"])
    .rename(columns={"mean": "Overall mean", "std": "Std"})
    .sort_values("Overall mean", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, max(3, len(overall) * 0.55)))
bars = ax.barh(
    [_label(s) for s in overall.index],
    overall["Overall mean"],
    xerr=overall["Std"],
    color=[PALETTE.get(s, "grey") for s in overall.index],
    capsize=5, alpha=0.85, error_kw=dict(elinewidth=1.5),
)
ax.set_xlim(0, 5.6)
ax.set_xlabel("Overall mean score (1–5)", fontsize=10)
ax.set_title("Overall Score Ranking", fontsize=12, fontweight="bold")
for bar, val in zip(bars, overall["Overall mean"]):
    ax.text(bar.get_width() + 0.08, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

overall.index = overall.index.map(_label)
print(overall.to_string())

## 9 · Pipetly model comparison

In [ ]:
pipetly_sources = [s for s in sources_present if s.startswith("pipetly_")]

if len(pipetly_sources) >= 2:
    x = np.arange(len(METRICS))
    width = min(0.8 / len(pipetly_sources), 0.35)
    fig, ax = plt.subplots(figsize=(13, 5))

    for i, src in enumerate(pipetly_sources):
        sub = df[df["source"] == src]
        means = [sub[m].mean() for m in METRICS]
        stds  = [sub[m].std()  for m in METRICS]
        offset = (i - (len(pipetly_sources) - 1) / 2) * width
        ax.bar(x + offset, means, width, yerr=stds, capsize=4,
               color=PALETTE[src], label=_label(src), alpha=0.85,
               error_kw=dict(elinewidth=1.2, ecolor="#333"))

    ax.set_xticks(x)
    ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], fontsize=10)
    ax.set_ylabel("Score (1–5)", fontsize=11)
    ax.set_ylim(0, 5.8)
    ax.axhline(y=5, color="#ccc", linestyle="--", linewidth=0.8)
    ax.set_title(f"Pipetly: Model Comparison ({len(pipetly_sources)} variants)",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print(f"Only {len(pipetly_sources)} Pipetly variant(s) found — skipping comparison.")

## 10 · Score stability across runs

In [ ]:
runs_present = sorted(df["run"].unique())
fig, ax = plt.subplots(figsize=(8, 4))

for src in sources_present:
    sub = df[df["source"] == src]
    if sub.empty:
        continue
    run_means = [sub[sub["run"] == r]["mean"].mean() for r in runs_present]
    ax.plot(runs_present, run_means, marker="o", linewidth=2,
            color=PALETTE[src], label=_label(src))

ax.set_xticks(runs_present)
ax.set_xticklabels([f"Run {r}" for r in runs_present])
ax.set_ylabel("Mean score (1–5)")
ax.set_ylim(1, 5.2)
ax.set_title("Score Stability across Independent Runs",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 11 · Justifications viewer

In [ ]:
SHOW_ENTRY = df["entry_id"].iloc[0]   # change to any entry_id
SHOW_RUN   = 1

sub = (
    df[(df["entry_id"] == SHOW_ENTRY) & (df["run"] == SHOW_RUN)]
    .set_index("source")
    .reindex([s for s in sources_present if s in df["source"].values])
    .reset_index()
)

for _, row in sub.iterrows():
    label = _label(row["source"])
    print(f"\n{'═'*65}")
    print(f"  {label}  |  Entry: {row['entry_id']}  |  Run {row['run']}  |  Mean: {row['mean']:.2f}")
    print(f"{'═'*65}")
    for m in METRICS:
        score = int(row[m])
        stars = '★' * score + '☆' * (5 - score)
        print(f"  {METRIC_LABELS[m]:22s} [{stars}] {score}/5")
        print(f"    → {row.get(f'{m}_just', '—')}")
    print()